In [45]:
import json
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

In [46]:
with open('documents-with-ids.json','rt') as f_in:
    documents = json.load(f_in)

In [47]:
model_name = 'multi-qa-MiniLM-L6-cos-v1'
model = SentenceTransformer(model_name)

/usr/local/python/3.12.1/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [48]:
for doc in tqdm(documents):
    question = doc['question']
    text = doc['text']

    qt = question + ' '+ text

    doc['question_vector'] = model.encode(question)
    doc['text_vector'] = model.encode(text)
    doc['question_text_vector'] = model.encode(qt)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [01:53<00:00,  8.38it/s]


In [49]:
df_ground_truth = pd.read_csv('groundtruth.csv')

In [50]:
ground_truth = df_ground_truth.to_dict(orient='records')
print(len(ground_truth))

4627


In [51]:
es_client = Elasticsearch('http://localhost:9200')
es_client.ping()

True

In [52]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "id": {"type": "keyword"},
            "question_vector":{
                "type": "dense_vector",
                "dims":384,
                "index": True,
                "similarity": "cosine"
            },
            "text_vector":{
                "type": "dense_vector",
                "dims":384,
                "index": True,
                "similarity": "cosine"
            },
            "question_text_vector":{
                "type": "dense_vector",
                "dims":384,
                "index": True,
                "similarity": "cosine"
            }
        }
    }
}

index_name = "course-questions"

es_client.indices.delete(index=index_name, ignore_unavailable=True)
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [53]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [00:25<00:00, 37.11it/s]


In [54]:
query = 'I just discovered the course. Can I still join it?'
course = "data-engineering-zoomcamp"

In [55]:
v_q = model.encode(query)

In [56]:
knn_query = {
     "field": "text_vector",
    "query_vector": v_q,
    "k": 5,
    "num_candidates": 10000,
    "boost": 0.5,
    "filter": {
        "term": {
            "course": course
        }
    }
}

In [57]:
keyword_query = {
    "bool": {
        "must": {
            "multi_match": {
                "query": query,
                "fields": ["question^3", "text", "section"],
                "type": "best_fields",
                "boost": 0.5,
            }
        },
        "filter": {
            "term": {
                "course": course
            }
        }
    }
}

In [58]:
response = es_client.search(
    index = index_name,
    query = keyword_query,
    knn = knn_query,
    size=5
)

In [ ]:
response["hits"]["hits"]

# Hybrid Search Pipeline

In [60]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [61]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

In [62]:
def elastic_search_hybrid(field,query,vector,course="data-engineering-zoomcamp",index_name="course-questions",size=5):
    knn_query = {
        "field": field,
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000,
        "boost": 0.5,
        "filter": {
            "term": {
                "course": course
            }
        }
    }
    keyword_query = {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question", "text", "section"],
                    "type": "best_fields",
                    "boost": 0.5,
                }
             },
            "filter": {
                "term": {
                    "course": course
                }
            }
        }
    }

    search_query = {
        "knn": knn_query,
        "query": keyword_query,
        "size": 5,
        "_source": ["text", "section", "question", "course", "id"]
    }

    es_results = es_client.search(
        index=index_name,
        body=search_query
    )

    result_docs = []

    for hit in es_results['hits']['hits']:
        result_docs.append(hit['_source'])

    return result_docs    

In [63]:
def question_hybrid(q):
    question = q['question']
    course = q['course']

    v_q = model.encode(question)

    return elastic_search_hybrid('question_vector', question, v_q, course)

In [64]:
def evaluate(ground_truth,search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)
    
    return f'{hit_rate(relevance_total)},{mrr(relevance_total)}'

In [65]:
evaluate(ground_truth,question_hybrid)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 4627/4627 [01:47<00:00, 42.94it/s]


'0.9234925437648585,0.8481665586052878'